# Demand Forecasting — IN00 Spare Parts (v2, refined)

This notebook is a thin wrapper over `forecast_pipeline.py`. All the modelling
logic lives in that module so it can be tested and reused. This version refines
the original `final.ipynb` in three areas:

**Correctness**
- No look-ahead leakage: `zero_rate`, `cv`, ADI and all rolling stats are
  recomputed *point-in-time* (history up to the prediction month only). The
  original computed them once over all 37 months and used them as features.
- Proper per-series MASE (each SKU scaled by its own naive error) instead of a
  single pooled scale.
- Honest metrics: **WAPE** is the headline KPI for intermittent demand; MAPE is
  reported only on non-zero actual months and clearly labelled.

**Accuracy**
- Rolling-origin backtest over the last 6 month-ends (not a single split).
- Model **routing by demand pattern** (Smooth / Erratic / Intermittent / Lumpy)
  with a robust fallback, plus an SBA+LightGBM **ensemble**.
- Richer point-in-time features (months-since-last-sale, ADI, non-zero count,
  trend ratio, rolling max, seasonal lag-12 ratio, class/cost).

**Inventory value**
- **Quantile forecasts** (P50/P90) and a **service-level safety stock** that
  uses each SKU's own target service level (`TSL`) — directly comparable to the
  existing ERP `SAFTY` / `REORDER_POINT`.


## 1 — Run the full pipeline

Point `data_file` at the real Excel. Use `light=True` for a faster smoke test.

In [ ]:
from forecast_pipeline import Config, run_pipeline

cfg = Config(
    data_file="../IN00_ST_DATA_3Years.xlsx",
    out_file="IN00_forecast_results_v2.xlsx",
    light=False,          # set True for a quick run
)
res = run_pipeline(cfg)

## 2 — Model leaderboard (backtest, Nov-25 → Apr-26)

Lower WAPE / vWAPE / MASE is better. MASE < 1 beats the naive last-month forecast.

In [ ]:
import pandas as pd
pd.set_option("display.width", 200)
res.summary.set_index("model")

## 3 — Best model per demand pattern + routing

In [ ]:
print("Routing (demand pattern -> chosen model):")
for k, v in res.routing.items():
    print(f"  {k:13s} -> {v}")

# Per-pattern WAPE table
res.by_pattern.pivot_table(index="demand_type", columns="model", values="WAPE")

## 4 — Hybrid (ours) vs the existing ERP system forecast

In [ ]:
import matplotlib.pyplot as plt

comp = pd.DataFrame({"Hybrid (ours)": res.hybrid_metrics,
                     "ERP FCST_H":    res.system_metrics})
display(comp.loc[["WAPE", "vWAPE", "MASE_med", "MAE", "RMSE", "Bias%"]])

ax = comp.loc[["WAPE", "vWAPE"]].plot.bar(figsize=(7, 4), rot=0)
ax.set_ylabel("error %  (lower is better)")
ax.set_title("Forecast accuracy: pattern-routed hybrid vs ERP system")
plt.tight_layout(); plt.show()

## 5 — Forward forecast + safety stock (Jun–Aug 2026)

`run_pipeline` refits on all history and writes per-SKU forward forecasts with
P50/P90 bands plus two safety-stock recommendations:
- `*_serviceLevel` — z·σ over the lead time, using each SKU's own `TSL`.
- `*_quantile`     — derived from the P90 demand band.

Both are shown next to the existing ERP `REORDER_POINT` / `SAFTY`.

In [ ]:
out = res.output
cols = ["PRODUCT", "MOVE_CLASS", "demand_type", "assigned_model",
        "fcst_Jun26", "p50_Jun26", "p90_Jun26",
        "target_service_level", "safety_stock_serviceLevel",
        "reorder_point_serviceLevel", "erp_reorder_point"]
out.sort_values("mean_demand", ascending=False)[cols].head(10)

## 6 — Notes & caveats

- The hybrid currently under-forecasts slightly (negative bias). That is normal
  for intermittent demand and is exactly what the **safety-stock layer** is for —
  size buffers from the P90 / service-level output, not the point forecast.
- The reorder-point comparison vs the ERP depends on the assumed
  `lead_time_months` (default 2.0) and the σ basis. Calibrate `lead_time_months`
  to your real replenishment lead times before acting on the deltas.
- All results are written to `IN00_forecast_results_v2.xlsx` (5 sheets).
